# Theoretische Grenzen der Bildregistrierung via Kreuzkorrelation

**Basierend auf:** Aguerrebere et al., *Fundamental Limits in Multi-image Alignment*, arXiv:1602.01541

---

## Signalmodell

Zwei Bilder: das Referenzbild $z_0(x) = u(x) + n_0(x)$ und ein verschobenes, verrauschtes Bild
$$z_1(x) = u(x - \tau) + n_1(x)$$
wobei $u$ das latente Bild, $\tau$ der unbekannte 2D-Shift und $n_i \sim \mathcal{N}(0, \sigma^2)$ additives weißes Gauß-Rauschen ist. Dies entspricht dem **paarweisen Fall** ($K=1$) des Papers.

## SNR-Definition

Das Paper definiert den SNR als Verhältnis von Gradientenenergie zu Rauschleistung:
$$\text{SNR} = \frac{1}{N W^2} \int S(\omega) \|\omega\|^2 \, d\omega$$
wobei $S(\omega)$ die spektrale Leistungsdichte des Bildes und $N = \sigma^2$ die Rauschvarianz ist, $W$ die Bandbreite.

Für **Flachspektrum-Bilder** ($S(\omega) = S_w$): $\quad \text{SNR}_w = S_w W^2 / (6N)$

Für **natürliche Bilder** ($S(\omega) = S_n / \|\omega\|^2$): $\quad \text{SNR}_n = S_n / N$

## Vier SNR-Regionen

| Region | SNR-Bereich | Verhalten |
|--------|-------------|----------|
| I – Sehr hohes SNR | $\text{SNR} > \text{SNR}_1$ | Linear mit SNR, unabhängig von $K$ – paarweise Registrierung optimal |
| II – Hohes SNR | $\text{SNR}_2 < \text{SNR} < \text{SNR}_1$ | Superlinear, mehr Bilder helfen |
| III – Übergang | $\text{SNR}_3 < \text{SNR} < \text{SNR}_2$ | Exponentieller Fehleranstieg |
| IV – Sättigung | $\text{SNR} < \text{SNR}_3$ | Keine Registrierung möglich |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.integrate import quad
from scipy.special import erfc

plt.rcParams.update({
    'font.size': 12,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

## Implementierung der Schranken

### 1. Cramér-Rao Schranke – deterministisches Bildmodell (CRBD)

Für eine unbekannte deterministische Unterlage $u$ gilt (Gl. 16):
$$\text{CRBD} = \sigma^2 \frac{\|u_x\|^2 + \|u_y\|^2}{\|u_x\|^2 \|u_y\|^2 - (u_x^T u_y)^2}$$

Für ein rotationssymmetrisches Bild (Hochspannungs-Näherung, Gl. 112):
$$\text{CRBD} \approx \frac{16\pi^2}{N_p W^2 \cdot \text{SNR}}$$

**Wichtig:** Diese Schranke ist **unabhängig von $K$** (Anzahl der Bilder). Sie ist nur bei hohem SNR tight.

Fall mit bekanntem $u$ (z.B. rauschfreie Referenz): $\text{CRBD}_{\text{kn}} = \text{CRBD}/2$

In [ ]:
def CRBD(SNR, Np, W=2*np.pi):
    """CRB deterministisches Modell (Hochspannungs-Näherung, Gl. 16/112).
    MSE in Pixel^2, unabhängig von K."""
    return 16 * np.pi**2 / (Np * W**2 * SNR)

def CRBD_kn(SNR, Np, W=2*np.pi):
    """CRBD für bekanntes u (rauschfreie Referenz), Gl. 19."""
    return CRBD(SNR, Np, W) / 2

### 2. Cramér-Rao Schranke – stochastisches Bildmodell (CRBS)

Für ein stationäres Gaußsches Bildmodell mit spektraler Dichte $S(\omega)$ und rotationssymmetrischem Spektrum (Gl. 40):
$$\text{CRBS} = \frac{2}{(K+1) \, \rho_{xx}}, \quad \rho_{xx} = \sum_l \frac{2 S^2(\omega_l) \omega_{lx}^2}{N^2 + (K+1) S(\omega_l) N}$$

**Flachspektrum** ($S(\omega) = S_w$, Gl. 50):
$$\text{CRBS}_w = \frac{8\pi^2 (W^2 + 6\,\text{SNR}_w(K+1))}{3 N_p (K+1) \text{SNR}_w^2 W^2}$$

**Natürliche Bilder** ($S(\omega) \propto \|\omega\|^{-2}$, aus Appendix C, Gl. 123):
$$\text{CRBS}_n = \frac{8\pi}{N_p (K+1) \text{SNR}_n^2 \cdot \operatorname{arccoth}\!\left(1 + \frac{2\pi(K+1)\,\text{SNR}_n}{W^2}\right)}$$

**Brechpunkt** (ab wo mehr Bilder helfen, Gl. 52/47):
$$\text{SNR}_1^{(K)} = \frac{W^2}{6(K+1)} \quad \text{(Flachspektrum)}$$

Beide Schranken konvergieren bei hohem SNR gegen $16\pi^2 / (N_p W^2 \,\text{SNR})$.

In [ ]:
def acoth(x):
    """Inverser hyperbolischer Kotangens: acoth(x) = arctanh(1/x) für |x|>1."""
    return np.arctanh(1.0 / x)

def CRBSw(SNR, K, Np, W=2*np.pi):
    """CRBS Flachspektrum (Gl. 50). MSE in Pixel^2."""
    return 8 * np.pi**2 * (W**2 + 6*SNR*(K+1)) / (3 * Np * (K+1) * SNR**2 * W**2)

def CRBSn(SNR, K, Np, W=2*np.pi):
    """CRBS natürliche Bilder (Appendix C, Gl. 123). MSE in Pixel^2."""
    arg = 1.0 + 2*np.pi*(K+1)*SNR / W**2
    return 8 * np.pi / (Np * (K+1) * SNR**2 * acoth(arg))

def SNR1_w(K, W=2*np.pi):
    """Brechpunkt SNR_1 für Flachspektrum (Gl. 52)."""
    return W**2 / (6*(K+1))

def SNR1_n(K, W=2*np.pi):
    """Brechpunkt SNR_1 für natürliche Bilder (Gl. 47)."""
    return W**2 / (2*np.pi*(K+1))

### 3. Extended Ziv-Zakai Schranke (EZZB)

Die EZZB ist eine tightere Bayes-Schranke, die den Fehler mit der Fehlerwahrscheinlichkeit eines binären Detektionsproblems verknüpft. Sie erfasst alle vier SNR-Regionen korrekt (Gl. 63–64, für $W=2\pi$, Flachspektrum):

$$\text{EZZB}_w = \frac{1}{c^2} \int_0^{\sqrt{2b}} h \, e^{-\frac{9h^4}{20 N_p}} \Phi(h) \, dh \;+\; \frac{D^2}{6} \, e^{a+b} \, \Phi(\sqrt{2b})$$

mit
$$\kappa_1 = \frac{9\,\text{SNR}_w^2 (K+1)}{8\pi^4 + 12\pi^2 \,\text{SNR}_w(K+1)}, \quad
\kappa_2 = \frac{9\,\text{SNR}_w^2 K}{4\pi^4 + 6\pi^2 \,\text{SNR}_w(K+1)}$$
$$a = -N_p \ln\!\frac{\sqrt{\kappa_2+1}+1}{2}, \quad
b = \frac{N_p}{2} \frac{\sqrt{\kappa_2+1}-1}{\sqrt{\kappa_2+1}}, \quad
c^2 = \frac{N_p \pi^2 \kappa_1}{12}$$

$\Phi(t) = \frac{1}{\sqrt{2\pi}} \int_t^\infty e^{-s^2/2} ds$ ist die Q-Funktion. $D$ ist die Breite des uniformen Shift-Priors.

Die EZZB gibt den EMSE **einer** Shift-Komponente in Pixel².

In [ ]:
def Q_func(h):
    """Q-Funktion: oberer Schwanz der Standardnormalverteilung."""
    return 0.5 * erfc(np.asarray(h) / np.sqrt(2))

def EZZBw_single(SNR, K, Np, D=10.0):
    """EZZB für Flachspektrum, W=2pi, eine Shift-Komponente (Gl. 63-64).
    Gibt EMSE in Pixel^2 zurück."""
    if SNR <= 0:
        return D**2 / 12.0

    kappa1 = 9 * SNR**2 * (K+1) / (8*np.pi**4 + 12*np.pi**2 * SNR*(K+1))
    kappa2 = 9 * SNR**2 * K     / (4*np.pi**4 +  6*np.pi**2 * SNR*(K+1))

    sq = np.sqrt(kappa2 + 1.0)
    a  = -Np * np.log((sq + 1.0) / 2.0)
    b  =  Np / 2.0 * (sq - 1.0) / sq
    c2 =  Np * np.pi**2 * kappa1 / 12.0

    upper = np.sqrt(max(2.0*b, 0.0))

    if c2 < 1e-300:
        term1 = 0.0
    else:
        def integrand(h):
            exp_arg = -9.0 * h**4 / (20.0 * Np)
            return h * np.exp(np.clip(exp_arg, -700, 0)) * Q_func(h)
        integral, _ = quad(integrand, 0.0, upper, limit=500, epsabs=1e-12, epsrel=1e-8)
        term1 = integral / c2

    ab = a + b
    term2 = D**2 / 6.0 * np.exp(np.clip(ab, -700, 0)) * Q_func(np.sqrt(max(2.0*b, 0.0)))

    return max(term1 + term2, 0.0)

def EZZBw(SNR_arr, K, Np, D=10.0):
    """Vektorisierte EZZB über ein SNR-Array."""
    return np.array([EZZBw_single(s, K, Np, D) for s in SNR_arr])

---
## Plot 1: CRBS vs. SNR – Einfluss der Bildgröße $N_p$

Für paarweise Registrierung ($K=1$), Flachspektrum. Die Schranke ist **linear in $N_p$**.

In [ ]:
SNR_dB = np.linspace(-25, 20, 500)
SNR    = 10**(SNR_dB / 10)
K      = 1
W      = 2*np.pi

Np_list   = [10**2, 20**2, 50**2, 100**2, 200**2]
Np_labels = ['$10\\times10$', '$20\\times20$', '$50\\times50$', '$100\\times100$', '$200\\times200$']
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(Np_list)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, title) in zip(axes, [
        ('Flachspektrum (CRBS$_w$, Gl. 50)', 'Flachspektrum'),
        ('Natürliche Bilder (CRBS$_n$, Appendix C)', 'Natürliche Bilder')
    ]):
    for Np, lbl, col in zip(Np_list, Np_labels, colors):
        if 'Flach' in title:
            rmse = np.sqrt(np.array([CRBSw(s, K, Np, W) for s in SNR]))
        else:
            rmse = np.sqrt(np.array([CRBSn(s, K, Np, W) for s in SNR]))
        ax.semilogy(SNR_dB, rmse, color=col, label=lbl)

    # CRBD (Referenz, K-unabhängig)
    for Np, col in zip(Np_list, colors):
        rmse_d = np.sqrt(CRBD(SNR, Np, W))
        ax.semilogy(SNR_dB, rmse_d, color=col, linestyle='--', alpha=0.5)

    # Brechpunkt SNR_1 für K=1
    snr1 = SNR1_w(K, W)
    ax.axvline(10*np.log10(snr1), color='gray', linestyle=':', label=f'SNR$_1$ ($K=1$)')

    ax.set_xlabel('SNR (dB)')
    ax.set_ylabel('RMSE (Pixel)')
    ax.set_title(title)
    ax.set_ylim(1e-4, 10)
    ax.legend(title='Bildgröße', fontsize=9)

axes[0].plot([], [], 'k-',  label='CRBS$_w$ (stochastisch)')
axes[0].plot([], [], 'k--', alpha=0.5, label='CRBD (deterministisch)')
axes[0].legend(title='Bildgröße', fontsize=9)

fig.suptitle('Einfluss der Bildgröße $N_p$ auf die Registrierungsschranken ($K=1$)', fontsize=13)
plt.tight_layout()
plt.savefig('plot1_bildgroesse.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Plot 2: CRBS vs. SNR – Einfluss der Bildstruktur (Spektrum)

Für $N_p = 50\times50$, $K=1$: Vergleich Flachspektrum vs. natürliche Bilder, und Einfluss der Bandbreite $W$.

Die Bandbreite $W$ bestimmt, wie viel hochfrequente Information das Bild enthält. Ein kleineres $W$ bedeutet ein glatteres (bandbegrenztes) Bild.

In [ ]:
Np = 50**2
K  = 1

W_list   = [np.pi/2, np.pi, 2*np.pi]
W_labels = ['$W=\\pi/2$ (glattes Bild)', '$W=\\pi$ (mittel)', '$W=2\\pi$ (volles Spektrum)']
colors_w = ['C0', 'C1', 'C2']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for W, lbl, col in zip(W_list, W_labels, colors_w):
    rmse_w = np.sqrt(np.array([CRBSw(s, K, Np, W) for s in SNR]))
    rmse_n = np.sqrt(np.array([CRBSn(s, K, Np, W) for s in SNR]))

    axes[0].semilogy(SNR_dB, rmse_w, color=col, label=lbl)
    axes[1].semilogy(SNR_dB, rmse_n, color=col, label=lbl)

    # Brechpunkte
    snr1 = SNR1_w(K, W)
    axes[0].axvline(10*np.log10(snr1), color=col, linestyle=':', alpha=0.6)

for ax, title in zip(axes, ['CRBS$_w$ – Flachspektrum', 'CRBS$_n$ – Natürliche Bilder']):
    ax.set_xlabel('SNR (dB)')
    ax.set_ylabel('RMSE (Pixel)')
    ax.set_title(title)
    ax.set_ylim(1e-4, 10)
    ax.legend(fontsize=9)
    ax.text(0.02, 0.05, f'$N_p = 50\\times50$, $K={K}$',
            transform=ax.transAxes, fontsize=9, bbox=dict(boxstyle='round', alpha=0.1))

fig.suptitle('Einfluss der Bildstruktur (Bandbreite $W$)', fontsize=13)
plt.tight_layout()
plt.savefig('plot2_bildstruktur.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Plot 3: Extended Ziv-Zakai Schranke (EZZB)

Die EZZB ist die tighteste bekannte Schranke und zeigt alle vier SNR-Regionen. Hier für Flachspektrum ($W=2\pi$).

**Variation der Bildgröße** und **Anzahl Bilder** $K$.

In [ ]:
SNR_dB_ezzb = np.linspace(-30, 15, 150)
SNR_ezzb    = 10**(SNR_dB_ezzb / 10)
D = 10.0  # Shift-Prior: uniform auf [0, D] Pixel

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Links: Variation Np, K=1 ---
ax = axes[0]
Np_list_ezzb = [10**2, 20**2, 50**2, 100**2]
labels_ezzb  = ['$10\\times10$', '$20\\times20$', '$50\\times50$', '$100\\times100$']
colors_e = plt.cm.plasma(np.linspace(0.15, 0.85, len(Np_list_ezzb)))

for Np, lbl, col in zip(Np_list_ezzb, labels_ezzb, colors_e):
    ezzb = EZZBw(SNR_ezzb, K=1, Np=Np, D=D)
    ax.semilogy(SNR_dB_ezzb, np.sqrt(ezzb), color=col, label=lbl)
    # CRBS für Vergleich
    crbs = np.array([CRBSw(s, 1, Np) for s in SNR_ezzb])
    ax.semilogy(SNR_dB_ezzb, np.sqrt(crbs), color=col, linestyle='--', alpha=0.4)

ax.axhline(D / np.sqrt(12), color='gray', linestyle=':', label=f'Sättigungswert $D/\\sqrt{{12}}$')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('RMSE (Pixel)')
ax.set_title('EZZB: Variation Bildgröße ($K=1$)')
ax.set_ylim(1e-3, 20)
ax.legend(title='Bildgröße', fontsize=9)
ax.plot([], [], 'k-',  label='EZZB')
ax.plot([], [], 'k--', alpha=0.4, label='CRBS$_w$')

# --- Rechts: Variation K, Np=50x50 ---
ax = axes[1]
Np = 50**2
K_list   = [1, 3, 10, 100]
K_labels = ['$K=1$ (paarweise)', '$K=3$', '$K=10$', '$K=100$']
colors_k = plt.cm.cool(np.linspace(0.1, 0.9, len(K_list)))

for Kv, lbl, col in zip(K_list, K_labels, colors_k):
    ezzb = EZZBw(SNR_ezzb, K=Kv, Np=Np, D=D)
    ax.semilogy(SNR_dB_ezzb, np.sqrt(ezzb), color=col, label=lbl)
    crbs = np.array([CRBSw(s, Kv, Np) for s in SNR_ezzb])
    ax.semilogy(SNR_dB_ezzb, np.sqrt(crbs), color=col, linestyle='--', alpha=0.4)

ax.axhline(D / np.sqrt(12), color='gray', linestyle=':', label=f'$D/\\sqrt{{12}}$')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('RMSE (Pixel)')
ax.set_title(f'EZZB: Variation Bildanzahl ($N_p = 50\\times50$, $D={D}$)')
ax.set_ylim(1e-3, 20)
ax.legend(fontsize=9)

fig.suptitle('Extended Ziv-Zakai Schranke (Flachspektrum, $W=2\\pi$)', fontsize=13)
plt.tight_layout()
plt.savefig('plot3_ezzb.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Plot 4: Gesamtvergleich aller Schranken (paarweise Registrierung, $K=1$)

Entspricht Abbildung 4 im Paper. Die vier SNR-Regionen sind farbig markiert.

In [ ]:
Np = 50**2
K  = 1
D  = 10.0
W  = 2*np.pi

SNR_dB_all = np.linspace(-30, 20, 200)
SNR_all    = 10**(SNR_dB_all / 10)

crbd  = np.sqrt(CRBD(SNR_all, Np, W))
crbsw = np.sqrt(np.array([CRBSw(s, K, Np, W) for s in SNR_all]))
crbsn = np.sqrt(np.array([CRBSn(s, K, Np, W) for s in SNR_all]))

SNR_dB_ez = np.linspace(-30, 15, 120)
SNR_ez    = 10**(SNR_dB_ez / 10)
ezzb = np.sqrt(EZZBw(SNR_ez, K, Np, D))

# SNR-Schwellwerte (näherungsweise für K=1)
snr1_dB = 10*np.log10(SNR1_w(K, W))   # ~2.2 dB für K=1, W=2pi

# SNR_2 und SNR_3 numerisch bestimmen aus EZZB-Verlauf
# SNR_2: Übergang – wo EZZB stark ansteigt (größte 2. Ableitung)
# SNR_3: Sättigung – wo EZZB die Sättigungsgrenze D/sqrt(12) erreicht
saturation = D / np.sqrt(12)

fig, ax = plt.subplots(figsize=(10, 6))

# Hintergrundfarben für die vier Regionen
ax.axvspan(-30,  snr1_dB - 5, alpha=0.06, color='blue')
ax.axvspan(snr1_dB - 5, snr1_dB, alpha=0.06, color='green')
ax.axvspan(snr1_dB, 20, alpha=0.06, color='orange')

ax.semilogy(SNR_dB_all, crbd,  'k--',  label='CRBD (deterministisch, $u$ unbekannt)', linewidth=1.5)
ax.semilogy(SNR_dB_all, crbsw, 'b-',   label='CRBS$_w$ (Flachspektrum)', linewidth=2)
ax.semilogy(SNR_dB_all, crbsn, 'g-.',  label='CRBS$_n$ (natürliche Bilder)', linewidth=2)
ax.semilogy(SNR_dB_ez,  ezzb,  'r-',   label='EZZB (Flachspektrum)', linewidth=2.5)
ax.axhline(saturation, color='gray', linestyle=':', linewidth=1.5,
           label=f'Sättigungswert $D/\\sqrt{{12}}$ = {saturation:.2f} Pixel')

ax.axvline(snr1_dB, color='gray', linestyle='--', alpha=0.7, linewidth=1)
ax.text(snr1_dB+0.3, 5, 'SNR$_1$', fontsize=9, color='gray')

# Annotations für die Regionen
ax.text(14,   3e-3, 'I\nSehr hohes SNR\n(paarweise optimal)', ha='center', fontsize=8, color='darkorange')
ax.text(snr1_dB-3, 3e-3, 'II\nHohes SNR',  ha='center', fontsize=8, color='darkgreen')
ax.text(-18,  0.3,  'III+IV\nÜbergang &\nSättigung', ha='center', fontsize=8, color='navy')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('RMSE (Pixel)', fontsize=12)
ax.set_title(f'Fundamentale Grenzen – Paarweise Registrierung ($K=1$, $N_p=50\\times50$, $D={D}$)',
             fontsize=12)
ax.set_xlim(-30, 20)
ax.set_ylim(1e-3, 10)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('plot4_vergleich.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Plot 5: Das theoretische Limit vs. SNR – Hauptplot

Kombinierte Parameterstudie: **Bildgröße** und **Bildstruktur** gemeinsam, paarweise ($K=1$).

In [ ]:
SNR_dB_main = np.linspace(-30, 20, 150)
SNR_main    = 10**(SNR_dB_main / 10)
K = 1
D = 10.0

configs = [
    # (Np, W, linestyle, label)
    (20**2,  2*np.pi, '-',  r'$20{\times}20$, $W=2\pi$ (weiß)'),
    (50**2,  2*np.pi, '-',  r'$50{\times}50$, $W=2\pi$ (weiß)'),
    (100**2, 2*np.pi, '-',  r'$100{\times}100$, $W=2\pi$ (weiß)'),
    (50**2,  np.pi,   '--', r'$50{\times}50$, $W=\pi$ (glatt)'),
    (50**2,  np.pi/2, ':',  r'$50{\times}50$, $W=\pi/2$ (sehr glatt)'),
]

cmap = plt.cm.tab10

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, bound_func, title in zip(
    axes,
    [CRBSw, CRBSn],
    ['CRBS$_w$ – Flachspektrum (Gl. 50)', 'CRBS$_n$ – Natürliche Bilder (App. C)']
):
    for i, (Np, W, ls, lbl) in enumerate(configs):
        vals = np.sqrt(np.array([bound_func(s, K, Np, W) for s in SNR_main]))
        ax.semilogy(SNR_dB_main, vals, color=cmap(i), linestyle=ls, label=lbl)

        # Brechpunkt markieren
        snr1 = SNR1_w(K, W)
        rmse_at_snr1 = np.sqrt(bound_func(snr1, K, Np, W))
        ax.plot(10*np.log10(snr1), rmse_at_snr1, 'o', color=cmap(i), markersize=5, alpha=0.7)

    ax.set_xlabel('SNR (dB)', fontsize=11)
    ax.set_ylabel('RMSE (Pixel)', fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(1e-4, 10)
    ax.set_xlim(-30, 20)
    ax.legend(fontsize=8, loc='upper right')
    ax.text(0.02, 0.97, 'Punkte = Brechpunkt SNR$_1$',
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', alpha=0.1))

fig.suptitle('Theoretisches Limit der paarweisen Bildregistrierung ($K=1$)\n'
             'Einfluss von Bildgröße und Bildstruktur', fontsize=13)
plt.tight_layout()
plt.savefig('plot5_parameterstudie.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Zusammenfassung der wichtigsten Ergebnisse

### Parameter und ihre Wirkung

| Parameter | Wirkung auf das Limit |
|-----------|----------------------|
| **Bildgröße $N_p$** | Schranke ∝ $1/N_p$ (linear); größere Bilder → bessere Registrierung |
| **SNR** | Zwei Regime: linear (hohes SNR) vs. superlinear (mittleres SNR) |
| **Bildstruktur (Bandbreite $W$)** | Engeres Spektrum → höherer RMSE im linearen Regime ($\propto 1/W^2$) |
| **Anzahl Bilder $K$** | Nur unterhalb SNR$_1$ hilfreich; schiebt SNR$_2$, SNR$_3$ nach unten |

### Kritische SNR-Schwellwerte (Flachspektrum, $W=2\pi$)

$$\text{SNR}_1^{(K=1)} = \frac{(2\pi)^2}{6 \cdot 2} = \frac{2\pi^2}{3} \approx 6.6 \approx 8.2\,\text{dB}$$

Oberhalb von SNR$_1$: paarweise Registrierung ist optimal – mehr Bilder bringen nichts.

### Fazit für die Praxis

1. **Hohes SNR** (> 8 dB für $K=1$): RMSE ∝ $1/(\sqrt{N_p \cdot W^2 \cdot \text{SNR}})$ – paarweise Kreuzkorrelation ist optimal
2. **Mittleres SNR**: Mehr Bilder helfen; Bildgröße und Spektrum sind entscheidend
3. **Niedriges SNR** (< ca. −5 dB für $N_p=50^2$): Kein zuverlässiges Alignment möglich
4. **Bildstruktur**: Bilder mit mehr Hochfrequenzanteil (großes $W$, hoher Gradient) sind leichter zu registrieren

---
## Anwendungsfall: Satellitenbildregistrierung gegen bekannte Referenz

**Rahmenbedingungen:**
- Modell: **CRBS$_n$** (natürliche Bilder, $S(\omega) \propto \|\omega\|^{-2}$)
- Referenz bekannt → **CRBD$_\text{kn}$ = CRBD/2** (nur ein Bild verrauscht, Gl. 19)
- SNR > 0 dB, Bildgröße $N_p \geq 100 \times 100$ Pixel
- $K = 1$ (paarweise Registrierung, Kreuzkorrelation mit Referenz)

Da SNR > 0 dB und $N_p$ groß ist, befindet sich die Anwendung sicher in **Region I** (sehr hohes SNR, paarweise optimal). CRBS$_n$ und CRBD$_\text{kn}$ sind dort nahezu identisch – die Wahl des Modells spielt kaum eine Rolle.

Das erreichbare Limit lautet:
$$\text{RMSE} \geq \sqrt{\text{CRBD}_\text{kn}} = \frac{2\sqrt{2}\,\pi}{\sqrt{N_p}\, W \sqrt{\text{SNR}}}$$

Der Faktor $1/2$ gegenüber dem unbekannten-Referenz-Fall kommt daher, dass nur **ein** Bild Rauschen trägt.

In [ ]:
from matplotlib.lines import Line2D

W = 2*np.pi
K = 1

SNR_dB_sat = np.linspace(0, 30, 300)
SNR_sat    = 10**(SNR_dB_sat / 10)

Np_sat     = [100**2, 200**2, 500**2, 1000**2]
lbl_sat    = ['$100\\times100$', '$200\\times200$', '$500\\times500$', '$1000\\times1000$']
colors_sat = plt.cm.magma(np.linspace(0.15, 0.75, len(Np_sat)))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Links: RMSE vs SNR ---
ax = axes[0]
for Np, lbl, col in zip(Np_sat, lbl_sat, colors_sat):
    rmse_n  = np.sqrt(np.array([CRBSn(s, K, Np, W) for s in SNR_sat]))
    rmse_kn = np.sqrt(CRBD_kn(SNR_sat, Np, W))
    ax.semilogy(SNR_dB_sat, rmse_n,  color=col, linestyle='-',  linewidth=2)
    ax.semilogy(SNR_dB_sat, rmse_kn, color=col, linestyle='--', linewidth=1.5)

# typische SNR-Referenzwerte
for snr_mark, label_mark in [(5, 'niedrig\n5 dB'), (15, 'typisch\n15 dB'), (25, 'sehr gut\n25 dB')]:
    ax.axvline(snr_mark, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    ax.text(snr_mark + 0.3, 0.25, label_mark, fontsize=7.5, color='dimgray', va='top')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('RMSE (Pixel)', fontsize=12)
ax.set_title('Registrierungslimit: CRBS$_n$ vs. CRBD$_{\\rm kn}$\n(natürl. Bilder, bekannte Referenz)', fontsize=11)
ax.set_xlim(0, 30)
ax.set_ylim(3e-5, 0.5)

legend_size  = [Line2D([0],[0], color=col, lw=2) for col in colors_sat]
legend_type  = [Line2D([0],[0], color='k', lw=2, ls='-'),
                Line2D([0],[0], color='k', lw=1.5, ls='--')]
ax.legend(legend_size + legend_type,
          lbl_sat + ['CRBS$_n$ (stoch., $u$ unbekannt)',
                     'CRBD$_{\\rm kn}$ (det., Ref. bekannt)'],
          fontsize=8.5, loc='upper right')

# --- Rechts: RMSE-Heatmap in Millipixel ---
ax2 = axes[1]
snr_vals = np.array([0, 5, 10, 15, 20, 25, 30])
Np_vals  = np.array([100**2, 200**2, 500**2, 1000**2, 2000**2])
Np_lbls  = ['$100\\times100$', '$200\\times200$', '$500\\times500$',
             '$1000\\times1000$', '$2000\\times2000$']

table = np.zeros((len(Np_vals), len(snr_vals)))
for i, Np in enumerate(Np_vals):
    for j, snr_db in enumerate(snr_vals):
        snr = 10**(snr_db / 10)
        table[i, j] = np.sqrt(CRBSn(snr, K, Np, W)) * 1000  # Millipixel

im = ax2.imshow(table, aspect='auto', cmap='RdYlGn_r',
                norm=plt.matplotlib.colors.LogNorm(vmin=0.05, vmax=200))
plt.colorbar(im, ax=ax2, label='RMSE (Millipixel)')

ax2.set_xticks(range(len(snr_vals)))
ax2.set_xticklabels([f'{s} dB' for s in snr_vals])
ax2.set_yticks(range(len(Np_vals)))
ax2.set_yticklabels(Np_lbls)
ax2.set_xlabel('SNR', fontsize=11)
ax2.set_ylabel('Bildgröße $N_p$', fontsize=11)
ax2.set_title('RMSE-Untergrenze [Millipixel]\n(CRBS$_n$, bekannte Referenz, $K=1$, $W=2\\pi$)', fontsize=11)

for i in range(len(Np_vals)):
    for j in range(len(snr_vals)):
        val = table[i, j]
        txt = f'{val:.1f}' if val >= 1.0 else f'{val:.2f}'
        color = 'white' if val > 20 else 'black'
        ax2.text(j, i, txt, ha='center', va='center', fontsize=8.5, color=color)

fig.suptitle('Fundamentales Registrierungslimit – Satellitenbild gegen bekannte Referenz',
             fontsize=13)
plt.tight_layout()
plt.savefig('plot6_satellit.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRMSE-Untergrenze in Pixel (CRBS_n, K=1, W=2pi):')
header = f'{"Bildgröße":>14}' + ''.join(f'  {s:2d} dB' for s in snr_vals)
print(header)
print('-' * len(header))
for i, (Np, lbl) in enumerate(zip(Np_vals, ['100²', '200²', '500²', '1000²', '2000²'])):
    row = f'{lbl:>14}' + ''.join(f'  {table[i,j]/1000:6.4f}' for j in range(len(snr_vals)))
    print(row + '  Pixel')

In [ ]:
# Schnellübersicht: Erreichbare RMSE für typische Parameter
print("Erreichbare RMSE-Untergrenze (CRBS_w, Flachspektrum, K=1, W=2pi)")
print("="*65)
print(f"{'SNR (dB)':>10} | {'10x10':>8} | {'50x50':>8} | {'100x100':>10} | {'200x200':>10}")
print("-"*65)
for snr_db in [-20, -10, -5, 0, 5, 10, 15, 20]:
    snr = 10**(snr_db/10)
    vals = [np.sqrt(CRBSw(snr, 1, n**2)) for n in [10, 50, 100, 200]]
    print(f"{snr_db:>10} | {vals[0]:>8.4f} | {vals[1]:>8.4f} | {vals[2]:>10.5f} | {vals[3]:>10.5f}")
print("\nWerte in Pixel (RMSE)")

---
## Numerisches Beispiel: Kreuzkorrelation auf simuliertem Naturbild (Sub-Pixel)

**Setup:**
- Bild $u$: simuliert mit $1/f^2$-Spektrum ($100 \times 100$ Pixel)
- Referenz: das **saubere** Bild $u$ (bekannte Referenz → CRBD$_\text{kn}$)
- Ziel-Bild: $z = u(x - \tau) + n$, mit Sub-Pixel-Shift $\tau = (3.7,\, 5.2)$ Pixel und AWGN
- **Shift-Erzeugung:** Fourier-Shift-Theorem — Multiplikation der FFT mit einer Phasenrampe $e^{-i2\pi(\omega_x \tau_x + \omega_y \tau_y)}$, dann IFFT. Exakt für bandbegrenzte Signale, **keinerlei räumliche Interpolation**.
- **Algorithmus:** Kreuzkorrelation via FFT + **Parabolische Interpolation** um das CC-Maximum → Sub-Pixel-Genauigkeit ohne Interpolationsfehler auf der Signalseite.
- Vergleich: empirischer RMSE (300 Monte-Carlo-Trials) gegen CRBD$_\text{kn}$ und CRBS$_n$

Der SNR wird nach der Paper-Definition aus der Gradientenenergie des konkreten Bildes berechnet:
$$\text{SNR} = \frac{\|u_x\|^2 + \|u_y\|^2}{N_p \, \sigma^2}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
N  = 100
Np = N * N
W  = 2 * np.pi

# ---- Simuliertes 1/f²-Bild ----
fy_vec = np.fft.fftfreq(N)
fx_vec = np.fft.fftfreq(N)
FX, FY = np.meshgrid(fx_vec, fy_vec)
F2 = FX**2 + FY**2
F2[0, 0] = 1.0

white      = np.random.randn(N, N) + 1j * np.random.randn(N, N)
spec       = white / np.sqrt(F2)
spec[0, 0] = 0.0
u = np.real(np.fft.ifft2(spec))
u = (u - u.mean()) / u.std()

# ---- Gradientenenergie (zirkuläre endliche Differenzen) ----
ux_img      = np.diff(u, axis=1, append=u[:, :1])
uy_img      = np.diff(u, axis=0, append=u[:1, :])
grad_energy = float(np.sum(ux_img**2 + uy_img**2))

print(f"Bild {N}×{N} Pixel, Np = {Np}")
print(f"Gradientenenergie: {grad_energy:.1f}")
print(f"SNR_paper bei σ = 1: {grad_energy/Np:.4f}  ({10*np.log10(grad_energy/Np):.1f} dB)")

# ---- Sub-Pixel-Shift via Fourier-Shift-Theorem (exakt, keine Interpolation) ----
def fourier_shift(img, dx, dy):
    """Exakter Sub-Pixel-Shift: Phasenrampe im Frequenzraum."""
    N   = img.shape[0]
    fx  = np.fft.fftfreq(N)
    fy  = np.fft.fftfreq(N)
    FX_, FY_ = np.meshgrid(fx, fy)
    return np.real(np.fft.ifft2(
        np.fft.fft2(img) * np.exp(-1j * 2 * np.pi * (FX_ * dx + FY_ * dy))
    ))

tau_x, tau_y = 3.7, 5.2
u_shifted = fourier_shift(u, tau_x, tau_y)

# ---- Visualisierung: Referenz + 3 verrauschte Zielbilder ----
SNR_dB_show = [15, 5, -5]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(u, cmap='gray', vmin=-3, vmax=3)
axes[0].set_title('Referenz $u$\n(sauber, 1/f²-Spektrum)', fontsize=10)
axes[0].axis('off')

for ax, snr_db in zip(axes[1:], SNR_dB_show):
    sigma = np.sqrt(grad_energy / (Np * 10**(snr_db / 10)))
    z     = u_shifted + np.random.randn(N, N) * sigma
    ax.imshow(z, cmap='gray', vmin=-3, vmax=3)
    ax.set_title(f'Zielbild $z$,  SNR = {snr_db} dB\n'
                 f'($\\sigma = {sigma:.2f}$,  Shift $({tau_x},{tau_y})$ px)', fontsize=10)
    ax.axis('off')

fig.suptitle(f'Simuliertes 1/f²-Bild (100×100 Pixel) – Sub-Pixel-Shift via Fourier-Shift-Theorem\n'
             f'($\\tau = ({tau_x},\\,{tau_y})$ px, keinerlei räumliche Interpolation)',
             fontsize=11)
plt.tight_layout()
plt.savefig('plot7a_bilder.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Sub-Pixel-Estimator: FFT-Kreuzkorrelation + Parabolischer Peak-Fit ----
def fft_xcorr_subpixel(ref, img, N):
    """Sub-Pixel-Shift via FFT-Kreuzkorrelation + Parabolischen Peak-Fit (3-Punkt)."""
    CC  = np.real(np.fft.ifft2(np.conj(np.fft.fft2(ref)) * np.fft.fft2(img)))
    idx = np.unravel_index(np.argmax(CC), CC.shape)
    r, c = idx

    def parab_offset(vm, v0, vp):
        d = 2 * (2*v0 - vm - vp)
        return (vp - vm) / d if abs(d) > 1e-12 else 0.0

    dr = parab_offset(CC[(r-1) % N, c], CC[r, c], CC[(r+1) % N, c])
    dc = parab_offset(CC[r, (c-1) % N], CC[r, c], CC[r, (c+1) % N])

    ty = (r + dr)
    tx = (c + dc)
    ty = ty if ty <= N / 2 else ty - N
    tx = tx if tx <= N / 2 else tx - N
    return tx, ty

# ---- Monte Carlo ----
np.random.seed(7)
SNR_dB_mc = np.linspace(-20, 20, 55)
SNR_mc    = 10 ** (SNR_dB_mc / 10)
n_trials  = 300

print(f"Monte Carlo: {n_trials} Trials × {len(SNR_dB_mc)} SNR-Punkte ...")
rmse_emp = np.zeros(len(SNR_dB_mc))

for j, snr in enumerate(SNR_mc):
    sigma   = np.sqrt(grad_energy / (Np * snr))
    sq_errs = np.zeros(n_trials)
    for t in range(n_trials):
        z          = u_shifted + np.random.randn(N, N) * sigma
        ex, ey     = fft_xcorr_subpixel(u, z, N)
        sq_errs[t] = ((ex - tau_x)**2 + (ey - tau_y)**2) / 2.0
    rmse_emp[j] = np.sqrt(sq_errs.mean())

print("Fertig.")

# Systematischer Bias des Parabolischen Fits (Plateau bei hohem SNR)
bias_floor = rmse_emp[-5:].mean()
print(f"Bias-Plateau (hoher SNR): {bias_floor*1000:.1f} Millipixel")

# ---- Theoretische Schranken (pro Komponente) ----
crbd_kn_arr = np.sqrt(CRBD_kn(SNR_mc, Np, W))
crbsn_arr   = np.sqrt([CRBSn(s, 1, Np, W) for s in SNR_mc])

# ---- Plot ----
fig, ax = plt.subplots(figsize=(11, 6))

ax.semilogy(SNR_dB_mc, rmse_emp,    'o-',  color='steelblue', markersize=4, linewidth=1.5,
            label=f'Empirisch – FFT + Parabolischer Fit ({n_trials} Trials)')
ax.semilogy(SNR_dB_mc, crbd_kn_arr, 'k--', linewidth=2.0,
            label='CRBD$_{\\rm kn}$ – bekannte Referenz (Gl. 19)')
ax.semilogy(SNR_dB_mc, crbsn_arr,   'g-.', linewidth=2.0,
            label='CRBS$_n$ – stochastisch, 1/f²-Modell (App. C)')

# Bias-Plateau
ax.axhline(bias_floor, color='tomato', linestyle=':', linewidth=1.5,
           label=f'Parabolischer Bias-Floor ≈ {bias_floor*1000:.0f} Millipixel')

# Zwei Regime annotieren
ax.text(-19, 0.35, 'Rauschbegrenzt\n(~CRB)', fontsize=9, color='navy',
        bbox=dict(boxstyle='round', alpha=0.08))
ax.text(7, 0.35, 'Bias-begrenzt\n(Parabolnäherung)', fontsize=9, color='darkred',
        bbox=dict(boxstyle='round', alpha=0.08))

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('RMSE (Pixel, pro Komponente)', fontsize=12)
ax.set_title(f'Monte Carlo vs. Theoretische Schranken\n'
             f'(100×100 Pixel, 1/f²-Spektrum, Sub-Pixel-Shift $\\tau=({tau_x},{tau_y})$ px, {n_trials} Trials)',
             fontsize=12)
ax.legend(fontsize=9.5, loc='upper right')
ax.set_xlim(-20, 20)
ax.set_ylim(5e-4, 5)

plt.tight_layout()
plt.savefig('plot7b_monte_carlo.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Zahlentabelle ----
print(f"\n{'SNR (dB)':>9} | {'RMSE emp.':>12} | {'CRBD_kn':>10} | {'CRBSn':>10} | {'emp/CRB':>8}")
print("-" * 60)
for idx in np.round(np.linspace(0, len(SNR_dB_mc)-1, 12)).astype(int):
    r = rmse_emp[idx] / crbd_kn_arr[idx]
    print(f"{SNR_dB_mc[idx]:>9.1f} | {rmse_emp[idx]:>12.5f} | "
          f"{crbd_kn_arr[idx]:>10.5f} | {crbsn_arr[idx]:>10.5f} | {r:>8.2f}×")

---
## Reales Beispiel: Meteosat Second Generation – IR_087 (8.7 µm)

**Datensatz:** zwei MSG2/SEVIRI Full-Disk Level-1B-Aufnahmen im Abstand von 15 Minuten  
(11:57 UTC und 12:12 UTC, 23. April 2026)

**Vorgehen:**
1. Wolkenfreien 128×128-Pixel-Ausschnitt (≈ 384×384 km bei 3 km/Pixel) automatisch suchen
2. Ausschnitt aus Bild 1 als saubere Referenz $u$ verwenden
3. Rauschpegel $\hat{\sigma}$ aus der zeitlichen Differenz der beiden Bilder schätzen: $\hat{\sigma} = \text{std}(z_2 - z_1)/\sqrt{2}$
4. Fourier-Shift $\tau = (0.7, 0.5)$ px + Monte-Carlo mit realem Bildinhalt und realistischem $\hat{\sigma}$
5. Vergleich empirischer RMSE mit CRBD$_\text{kn}$ für diesen konkreten Bildausschnitt

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from satpy import Scene
import glob, os

W  = 2 * np.pi

# ---- Dateien laden ----
nat_files = sorted(glob.glob('data/**/*.nat', recursive=True))
print("Dateien:")
for f in nat_files:
    print(" ", os.path.basename(f))

scn1 = Scene(filenames=[nat_files[0]], reader='seviri_l1b_native')
scn2 = Scene(filenames=[nat_files[1]], reader='seviri_l1b_native')
scn1.load(['IR_087']); scn2.load(['IR_087'])
arr1 = scn1['IR_087'].values.astype(np.float32)
arr2 = scn2['IR_087'].values.astype(np.float32)
print(f"\nFull-Disk Shape: {arr1.shape}  ({arr1.shape[0]*arr1.shape[1]/1e6:.1f} MPix)")
print(f"BT-Bereich Bild 1: {np.nanmin(arr1):.1f} … {np.nanmax(arr1):.1f} K")

# ---- Wolkenfreien 128×128-Ausschnitt suchen ----
# Kriterium: min BT > 275 K in beiden Bildern + maximale Textur (Gradientenenergie)
patch = 128
step  = 64
best_score, best_rc = -1, None

for r in range(0, arr1.shape[0] - patch, step):
    for c in range(0, arr1.shape[1] - patch, step):
        p1 = arr1[r:r+patch, c:c+patch]
        p2 = arr2[r:r+patch, c:c+patch]
        if np.any(np.isnan(p1)) or np.any(np.isnan(p2)):
            continue
        if p1.min() < 275 or p2.min() < 275:
            continue
        gx = np.diff(p1, axis=1)
        gy = np.diff(p1, axis=0)
        score = float(np.sum(gx**2) + np.sum(gy**2))
        if score > best_score:
            best_score, best_rc = score, (r, c)

r0, c0 = best_rc
u_msg   = arr1[r0:r0+patch, c0:c0+patch].astype(np.float64)
z2_msg  = arr2[r0:r0+patch, c0:c0+patch].astype(np.float64)

# Gradientenenergie des Ausschnitts (für exakte CRBD_kn)
ux_msg = np.diff(u_msg, axis=1, append=u_msg[:, :1])
uy_msg = np.diff(u_msg, axis=0, append=u_msg[:1, :])
grad_energy_msg = float(np.sum(ux_msg**2) + np.sum(uy_msg**2))
Np_msg = patch * patch

# Rauschschätzung aus zeitlicher Differenz: sigma = std(diff) / sqrt(2)
diff_patch = z2_msg - u_msg
sigma_est  = float(np.std(diff_patch) / np.sqrt(2))
snr_est    = grad_energy_msg / (Np_msg * sigma_est**2)
print(f"\nGewählter Ausschnitt: Zeile {r0}, Spalte {c0}  ({patch}×{patch} Pixel)")
print(f"BT-Bereich  Bild 1: {u_msg.min():.1f} … {u_msg.max():.1f} K  (Mittel: {u_msg.mean():.1f} K)")
print(f"BT-Bereich  Bild 2: {z2_msg.min():.1f} … {z2_msg.max():.1f} K")
print(f"Gradientenenergie:   {grad_energy_msg:.0f}  K²/Pixel")
print(f"σ̂ (aus Δt-Differenz): {sigma_est:.3f} K")
print(f"SNR_paper:           {snr_est:.3f}  ({10*np.log10(snr_est):.1f} dB)")

# ---- Visualisierung ----
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

im0 = axes[0].imshow(u_msg,  cmap='gray', vmin=275, vmax=315)
axes[0].set_title(f'Bild 1  (11:57 UTC)\n[Zeile {r0}, Spalte {c0}]', fontsize=10)
plt.colorbar(im0, ax=axes[0], label='BT (K)', fraction=0.046)

im1 = axes[1].imshow(z2_msg, cmap='gray', vmin=275, vmax=315)
axes[1].set_title('Bild 2  (12:12 UTC)', fontsize=10)
plt.colorbar(im1, ax=axes[1], label='BT (K)', fraction=0.046)

vmax_diff = max(abs(diff_patch).max(), 0.5)
im2 = axes[2].imshow(diff_patch, cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
axes[2].set_title(f'Differenz Bild2 − Bild1\nstd = {np.std(diff_patch):.3f} K', fontsize=10)
plt.colorbar(im2, ax=axes[2], label='ΔBT (K)', fraction=0.046)

fig.suptitle('MSG2/SEVIRI IR_087 (8.7 µm) – wolkenfreier Ausschnitt (128×128 Pixel ≈ 384×384 km)',
             fontsize=11)
plt.tight_layout()
plt.savefig('plot8a_msg_patch.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Monte Carlo auf realem MSG-Ausschnitt ----
# Referenz u_msg (Bild 1), bekannte Referenz-Szenario:
#   z = fourier_shift(u_msg, tau) + noise(sigma)
# sigma variiert über den SNR-Bereich; als Orientierung dient sigma_est aus der Δt-Differenz.

np.random.seed(13)
tau_x_msg, tau_y_msg = 0.7, 0.5        # bekannter Sub-Pixel-Shift
SNR_dB_msg = np.linspace(-15, 25, 55)
SNR_msg    = 10 ** (SNR_dB_msg / 10)
n_trials   = 300
N_msg      = patch

rmse_msg = np.zeros(len(SNR_dB_msg))

# Fourier-Shift des realen Bildes (einmalig)
u_msg_shifted = fourier_shift(u_msg, tau_x_msg, tau_y_msg)

print(f"Monte Carlo auf MSG-Ausschnitt: {n_trials} Trials × {len(SNR_dB_msg)} SNR-Punkte ...")
for j, snr in enumerate(SNR_msg):
    sigma   = np.sqrt(grad_energy_msg / (Np_msg * snr))
    sq_errs = np.zeros(n_trials)
    for t in range(n_trials):
        z          = u_msg_shifted + np.random.randn(N_msg, N_msg) * sigma
        ex, ey     = fft_xcorr_subpixel(u_msg, z, N_msg)
        sq_errs[t] = ((ex - tau_x_msg)**2 + (ey - tau_y_msg)**2) / 2.0
    rmse_msg[j] = np.sqrt(sq_errs.mean())

print("Fertig.")

# Bias-Plateau bei hohem SNR
bias_msg = rmse_msg[-5:].mean()
print(f"Bias-Plateau: {bias_msg*1000:.1f} Millipixel")

# Theoretische Schranken für diesen Ausschnitt
crbd_kn_msg = np.sqrt(CRBD_kn(SNR_msg, Np_msg, W))
crbsn_msg   = np.sqrt([CRBSn(s, 1, Np_msg, W) for s in SNR_msg])

# Betriebspunkt: geschätztes SNR aus den echten Daten
snr_op_dB  = 10 * np.log10(snr_est)
rmse_op_kn = np.sqrt(CRBD_kn(snr_est, Np_msg, W))

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Links: Monte Carlo vs. Schranken ---
ax = axes[0]
ax.semilogy(SNR_dB_msg, rmse_msg,     'o-',  color='firebrick',  markersize=4, linewidth=1.5,
            label=f'Empirisch – FFT + Parabolischer Fit ({n_trials} Trials)')
ax.semilogy(SNR_dB_msg, crbd_kn_msg,  'k--', linewidth=2.0,
            label='CRBD$_{\\rm kn}$ – bekannte Referenz (Gl. 19)')
ax.semilogy(SNR_dB_msg, crbsn_msg,    'g-.', linewidth=2.0,
            label='CRBS$_n$ – 1/f²-Modell (App. C)')
ax.axhline(bias_msg, color='tomato', linestyle=':', linewidth=1.5,
           label=f'Bias-Floor ≈ {bias_msg*1000:.0f} Millipixel')

# Betriebspunkt aus realen Daten markieren
ax.axvline(snr_op_dB, color='navy', linestyle='--', alpha=0.7, linewidth=1.5,
           label=f'Geschätztes SNR = {snr_op_dB:.1f} dB')
ax.plot(snr_op_dB, rmse_op_kn, 'D', color='navy', markersize=8, zorder=5,
        label=f'CRBD$_{{\\rm kn}}$ @ SNR = {rmse_op_kn*1000:.0f} Millipixel')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('RMSE (Pixel, pro Komponente)', fontsize=12)
ax.set_title('Monte Carlo vs. Theoretische Schranken\n'
             f'MSG IR_087, 128×128 Pixel, $\\tau=({tau_x_msg},{tau_y_msg})$ px',
             fontsize=11)
ax.legend(fontsize=8.5, loc='upper right')
ax.set_xlim(-15, 25)
ax.set_ylim(1e-4, 5)

# --- Rechts: Kreuzkorrelationsfeld (reale Bilder, zeitliche Differenz) ---
ax2 = axes[1]
CC_real = np.real(np.fft.ifft2(
    np.conj(np.fft.fft2(u_msg)) * np.fft.fft2(z2_msg)
))
CC_real_shift = np.fft.fftshift(CC_real)

# Nur den zentralen ±8-Pixel-Bereich zeigen
cx, cy = N_msg // 2, N_msg // 2
hw = 8
cc_crop = CC_real_shift[cy-hw:cy+hw+1, cx-hw:cx+hw+1]
ext = [-hw-0.5, hw+0.5, hw+0.5, -hw+0.5]
im = ax2.imshow(cc_crop, cmap='hot', extent=ext, aspect='equal', origin='upper')
plt.colorbar(im, ax=ax2, label='CC-Wert', fraction=0.046)

# Sub-Pixel-Peak aus den realen Bildern
ex_real, ey_real = fft_xcorr_subpixel(u_msg, z2_msg, N_msg)
ax2.plot(ex_real, ey_real, 'c+', markersize=14, markeredgewidth=2,
         label=f'Peak: $\\Delta x={ex_real:.3f}$, $\\Delta y={ey_real:.3f}$ px')
ax2.axvline(0, color='white', alpha=0.4, linewidth=0.8)
ax2.axhline(0, color='white', alpha=0.4, linewidth=0.8)
ax2.set_xlabel('$\\Delta x$ (Pixel)', fontsize=11)
ax2.set_ylabel('$\\Delta y$ (Pixel)', fontsize=11)
ax2.set_title(f'Kreuzkorrelation Bild1 vs. Bild2 (real)\n'
              f'Gemessener Offset: ({ex_real:.3f}, {ey_real:.3f}) Pixel', fontsize=11)
ax2.legend(fontsize=9, loc='upper right')

fig.suptitle('MSG2/SEVIRI IR_087 – Reales Registrierungsexperiment (128×128 Pixel)',
             fontsize=12)
plt.tight_layout()
plt.savefig('plot8b_msg_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nReale Kreuzkorrelation Bild1 vs. Bild2:")
print(f"  Gemessener Offset: Δx = {ex_real:.4f} px,  Δy = {ey_real:.4f} px")
print(f"  Erwartete RMSE-Untergrenze (CRBD_kn @ SNR = {snr_op_dB:.1f} dB): {rmse_op_kn*1000:.1f} Millipixel")
print(f"  → Der gemessene Offset liegt {'innerhalb' if abs(ex_real) < 3*rmse_op_kn and abs(ey_real) < 3*rmse_op_kn else 'außerhalb'} des 3σ-CRB-Bereichs")

---
## Warum funktioniert Matching noch bei negativem SNR?

Der Paper-SNR klingt klein, weil er auf **Pro-Pixel-Basis** definiert ist. Die Kreuzkorrelation wirkt aber als **Matched Filter** — sie summiert den Beitrag aller $N_p$ Pixel kohärent auf.

### CC-Peak vs. Rauschuntergrund

Der CC-Wert am korrekten Shift $\tau_0$ ist:
$$\text{CC}(\tau_0) = \sum_k u(k)\cdot z(k-\tau_0) \approx \|u\|^2 = N_p\cdot\text{Var}(u)$$

An einem falschen Shift $\tau \neq \tau_0$ ist die Erwartung null, die Streuung:
$$\text{std}\bigl[\text{CC}(\tau)\bigr] = \sigma\cdot\|u\| = \sigma\cdot\sqrt{N_p\cdot\text{Var}(u)}$$

Das **Detektions-SNR am CC-Peak** ist damit:
$$\text{SNR}_\text{Det} = \frac{N_p\cdot\text{Var}(u)}{\sigma\cdot\sqrt{N_p\cdot\text{Var}(u)}} = \sqrt{N_p}\cdot\frac{\text{std}(u)}{\sigma} = \sqrt{N_p}\cdot\sqrt{\text{SNR}_I}$$

Die Integration über $N_p$ Pixel hebt das Detektions-SNR also um den Faktor $\sqrt{N_p}$ an — für ein $128\times128$-Bild sind das **+42 dB** gegenüber dem Per-Pixel-Intensitäts-SNR.

### Beziehung der SNR-Definitionen

| SNR | Definition | Interpretation |
|-----|-----------|----------------|
| $\text{SNR}_\text{paper}$ | $(\|u_x\|^2+\|u_y\|^2)/(N_p\sigma^2)$ | Gradientenenergie pro Pixel / Rauschleistung |
| $\text{SNR}_I$ | $\text{Var}(u)/\sigma^2$ | Intensitätsvarianz / Rauschleistung |
| $\text{SNR}_\text{Det}$ | $\sqrt{N_p\cdot\text{SNR}_I}$ | Detektierbarkeit des CC-Peaks |

Für ein **1/f²-Bild** gilt: $\text{SNR}_I \approx \text{SNR}_\text{paper}\cdot\tfrac{N_p\cdot\text{Var}(u)}{\text{grad\_energy}}$  
(Konversionsfaktor > 1, da die Intensitätsleistung bei tiefen Frequenzen konzentriert ist, während die Gradientenenergie das hochfrequente Spektrum gewichtet).

Die CRB-Schranken kodieren denselben Effekt: sie skalieren mit $1/\sqrt{N_p}$, weil größere Bilder das effektive SNR proportional erhöhen.

In [ ]:
# ---- SNR-Vergleich für den MSG-Ausschnitt ----
W  = 2 * np.pi
Np_msg = 128**2

snr_paper_dB_range = np.linspace(-40, 15, 300)
snr_paper_range    = 10 ** (snr_paper_dB_range / 10)

# Konversionsfaktor Var(u) / (grad_energy/Np)  — bildspezifisch
conv = float(np.var(u_msg) / (grad_energy_msg / Np_msg))
snr_I_range   = snr_paper_range * conv
snr_det_range = np.sqrt(Np_msg * snr_I_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Links: SNR-Übersicht ---
ax = axes[0]
ax.plot(snr_paper_dB_range, 10*np.log10(snr_I_range),
        'C1-', linewidth=2, label=f'SNR$_I$ = SNR$_\\text{{paper}}$ + {10*np.log10(conv):.1f} dB')
ax.plot(snr_paper_dB_range, 10*np.log10(snr_det_range),
        'C2-', linewidth=2, label=f'SNR$_\\text{{Det}}$ = $\\sqrt{{N_p}}$ · SNR$_I^{{1/2}}$')
ax.axhline(0,  color='gray', linestyle=':', linewidth=1)
ax.axhline(10, color='gray', linestyle=':', linewidth=1, alpha=0.5)

# Betriebspunkt MSG
ax.axvline(10*np.log10(snr_est), color='navy', linestyle='--', linewidth=1.5,
           label=f'MSG-Betriebspunkt ({10*np.log10(snr_est):.1f} dB)')
ax.scatter([10*np.log10(snr_est)],
           [10*np.log10(snr_est * conv)], color='C1', s=60, zorder=5)
ax.scatter([10*np.log10(snr_est)],
           [10*np.log10(np.sqrt(Np_msg * snr_est * conv))], color='C2', s=60, zorder=5)

ax.set_xlabel('SNR$_\\text{paper}$ (dB)', fontsize=12)
ax.set_ylabel('SNR (dB)', fontsize=12)
ax.set_title('SNR-Konversion: Pro-Pixel → Detektion\n'
             f'(MSG IR_087, 128×128 Pixel, Konversionsfaktor = {10*np.log10(conv):.1f} dB)',
             fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(-40, 15)
ax.set_ylim(-30, 50)
ax.text(-39, 1.5, 'SNR = 0 dB', fontsize=8, color='gray')
ax.text(-39, 11.5, 'SNR = 10 dB', fontsize=8, color='gray')

# --- Rechts: Detektions-SNR und Fehlerrate ----
# P(Fehler) ≈ Np * Q(SNR_Det)  — Wahrscheinlichkeit, dass ein falscher Peak höher ist
from scipy.special import erfc
Q = lambda x: 0.5 * erfc(x / np.sqrt(2))

p_err = np.clip(Np_msg * Q(snr_det_range), 0, 1)

ax2 = axes[1]
ax2.semilogy(snr_paper_dB_range, p_err + 1e-10, 'C3-', linewidth=2,
             label='P(Fehler) ≈ $N_p \\cdot Q(\\text{SNR}_\\text{Det})$')
ax2.axhline(0.01, color='gray', linestyle=':', linewidth=1)
ax2.axhline(1e-6, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax2.axvline(10*np.log10(snr_est), color='navy', linestyle='--', linewidth=1.5,
            label=f'MSG-Betriebspunkt ({10*np.log10(snr_est):.1f} dB)')
ax2.text(-38, 0.015, '1% Fehlerrate', fontsize=8, color='gray')
ax2.text(-38, 1.5e-6, '10⁻⁶ Fehlerrate', fontsize=8, color='gray')

ax2.set_xlabel('SNR$_\\text{paper}$ (dB)', fontsize=12)
ax2.set_ylabel('Fehlerwahrscheinlichkeit', fontsize=12)
ax2.set_title('Fehlerwahrscheinlichkeit des CC-Argmax\n'
              f'($N_p = 128\\times128 = {Np_msg}$ Pixel)', fontsize=11)
ax2.legend(fontsize=10)
ax2.set_xlim(-40, 15)
ax2.set_ylim(1e-10, 2)

# Zeige den Betriebspunkt-Wert
p_op = float(Np_msg * Q(np.sqrt(Np_msg * snr_est * conv)))
ax2.scatter([10*np.log10(snr_est)], [max(p_op, 1e-10)], color='navy', s=60, zorder=5)
ax2.annotate(f'P ≈ {p_op:.2e}', xy=(10*np.log10(snr_est), max(p_op, 1e-10)),
             xytext=(10*np.log10(snr_est)+3, 1e-8), fontsize=9, color='navy',
             arrowprops=dict(arrowstyle='->', color='navy'))

fig.suptitle('Kreuzkorrelation als Matched Filter: Warum SNR$_\\text{paper}$ < 0 dB kein Problem ist',
             fontsize=12)
plt.tight_layout()
plt.savefig('plot9_snr_detection.png', dpi=150, bbox_inches='tight')
plt.show()

# Zahlentabelle
print(f"{'SNR_paper':>10} | {'sigma':>8} | {'SNR_I':>8} | {'SNR_Det':>10} | {'P(Fehler)':>12}")
print('-' * 58)
for snr_db in [10, 5, 0, -5, -10, -20, -30, -38]:
    snr  = 10**(snr_db/10)
    sig  = np.sqrt(grad_energy_msg / (Np_msg * snr))
    sI   = np.var(u_msg) / sig**2
    sD   = np.sqrt(Np_msg * sI)
    pe   = min(Np_msg * float(Q(sD)), 1.0)
    print(f"{snr_db:>8} dB | {sig:>7.1f} K | {10*np.log10(sI):>6.1f} dB | {sD:>10.1f} | {pe:>12.2e}")